# morphology_map

GA-SaMMT Map step (Step 1): threshold a derived raster (TPI, Openness, …)
into bathymetric high/low feature polygons, per the mean ± C·std rule
given in Huang et al. (2023) Eq. 2–5. One `MorphologyMethod` marker type
per algorithm selects which raster/sign to use; `threshold_mask` and
`polygonize_filtered` are the shared primitives every method funnels
through.

In [ ]:
#| hide
using MIM_GA

In [ ]:
#| default_exp morphology_map

In [ ]:
#| export
using Rasters
import ArchGDAL
using Statistics
using DataFrames

In [ ]:
#| export
# Marker types selecting which derived raster (and sign convention) a
# morphological_high/morphological_low call should use. One instance per
# algorithm (TPIMethod(), OpennessMethod(), ...); adding a new algorithm
# later means adding a new marker type plus its own morphological_high/low
# method pair below -- see LMIMethod/CIMethod, not yet implemented.
abstract type MorphologyMethod end
struct TPIMethod <: MorphologyMethod end
struct OpennessMethod <: MorphologyMethod end
struct BPIMethod      <: MorphologyMethod end

In [ ]:
#| export
"""
    threshold_mask(raster, C; sign) -> BitMatrix

The mean +/- C*std threshold rule shared by every GA-SaMMT Map-step tool
(Huang et al. 2023 Eq. 2-5, 6, 8): compute the spatial mean and standard
deviation of `raster` over its non-NaN cells, then select cells that are
`> mean + C*std` (`sign = :above`) or `< mean - C*std` (`sign = :below`).
NaN cells are never selected. An all-NaN raster returns an all-false mask
rather than erroring (mean/std of an empty set is undefined).
"""
function threshold_mask(raster::AbstractMatrix{<:Real}, C::Real; sign::Symbol)
    sign in (:above, :below) || throw(ArgumentError("sign must be :above or :below, got :$sign"))
    valid = .!isnan.(raster)
    mask = falses(size(raster))
    any(valid) || return mask

    m = mean(raster[valid])
    s = std(raster[valid])
    threshold = sign === :above ? m + C * s : m - C * s

    @inbounds for i in eachindex(raster)
        valid[i] || continue
        mask[i] = sign === :above ? raster[i] > threshold : raster[i] < threshold
    end
    return mask
end

In [ ]:
#| export
"""
    polygonize_filtered(mask, template::Raster, area_threshold) -> DataFrame

Convert a boolean mask into real-world polygons via GDAL's own raster
polygonizer (`GDALPolygonize`, through `ArchGDAL`/GDAL's C API directly --
ArchGDAL has no high-level raster-to-polygon wrapper as of v0.10, and the
pure-Julia alternative (GeometryOps.jl) has a known coordinate-offset bug
against Rasters.jl's `Intervals` sampling, so this goes through GDAL
itself rather than either). `template` supplies the mask's georeferencing
(dims, CRS): the mask is round-tripped through a `/vsimem/` in-memory GDAL
file via `Rasters.write` so GDAL derives the geotransform itself instead
of it being hand-computed here (hand-deriving it from `Points` vs.
`Intervals` sampling is exactly where the GeometryOps.jl bug above comes
from). Only polygons over `true` cells (GDAL's `DN=1`) with area at least
`area_threshold` are kept; the background (`DN=0`) polygon is always
dropped. Geometries are cloned before being collected -- `ArchGDAL`'s
layer-owned geometry becomes invalid once iteration moves past it.

Returns a `DataFrame` with one row per feature polygon (`:geometry`,
`:area`), not a bare geometry vector: this is the Map-step output that
the Characterise step (GA-SaMMT Tables 2-4: shape/topographic/profile
attributes) will attach further columns to, one attribute per column,
so it needs to already be a table rather than something restructured
into one later.

The output vector layer (and therefore every returned geometry) is
explicitly given `template`'s own CRS via `spatialref` -- `ArchGDAL`'s
`createlayer` does *not* default to the source raster's CRS, so omitting
this silently produces geometries with no CRS at all even though their
coordinate *values* are correct (this was missed in the first version:
the numbers were right, nothing declared what they meant). The CRS is
also attached as table-level metadata under the `"GEOINTERFACE:crs"` key
(plus a plain `"crs"` alias) -- the exact convention `GeoDataFrames.jl`
itself uses (see its `src/utils.jl`/`src/io.jl`), so this plain
`DataFrame` is discoverable by the wider GeoInterface/Tables.jl ecosystem
the same way a real `GeoDataFrame` would be, without taking on that
package's heavier dependency chain (GeometryOps, Proj, ...) for what is,
right now, just a CRS tag. If `template` itself has no CRS set, GDAL
round-trips it as a `LOCAL_CS["unnamed", ...]` placeholder rather than
erroring, so this still works (verified), it just tags the table as
explicitly CRS-less instead of silently so.
"""
function polygonize_filtered(mask::AbstractMatrix{Bool}, template::Raster, area_threshold::Real)
    area_threshold >= 0 || throw(ArgumentError("area_threshold must be >= 0, got $area_threshold"))
    size(mask) == size(template) || throw(DimensionMismatch(
        "mask size $(size(mask)) must match template raster size $(size(template))"))

    mask_raster = Rasters.rebuild(template; data=Int32.(mask), name=:mask, missingval=Int32(-1))
    vsimem_path = "/vsimem/mim_ga_poly_$(time_ns())_$(rand(UInt32)).tif"
    Rasters.write(vsimem_path, mask_raster; force=true)

    geoms = ArchGDAL.IGeometry[]
    areas = Float64[]
    crs_wkt = ""
    dataset = ArchGDAL.unsafe_read(vsimem_path)
    try
        band = ArchGDAL.getband(dataset, 1)
        srs = ArchGDAL.importWKT(ArchGDAL.getproj(dataset))
        crs_wkt = ArchGDAL.toWKT(srs)
        vec_ds = ArchGDAL.unsafe_create(ArchGDAL.getdriver("Memory"); filename="")
        try
            layer = ArchGDAL.createlayer(name="polys", dataset=vec_ds, geom=ArchGDAL.wkbPolygon,
                                          spatialref=srs)
            ArchGDAL.addfielddefn!(layer, "DN", ArchGDAL.OFTInteger)
            result = ArchGDAL.GDAL.gdalpolygonize(
                band.ptr, C_NULL, layer.ptr, 0, C_NULL, C_NULL, C_NULL,
            )
            result == ArchGDAL.GDAL.CE_None || error("GDALPolygonize failed with CPLErr=$result")

            for feature in layer
                ArchGDAL.getfield(feature, "DN") == 1 || continue  # drop the DN=0 background polygon
                g = ArchGDAL.getgeom(feature)
                a = ArchGDAL.geomarea(g)
                a >= area_threshold || continue
                push!(geoms, ArchGDAL.clone(g))
                push!(areas, a)
            end
        finally
            ArchGDAL.destroy(vec_ds)
        end
    finally
        ArchGDAL.destroy(dataset)
        ArchGDAL.GDAL.vsiunlink(vsimem_path)
    end

    df = DataFrame(geometry=geoms, area=areas)
    DataFrames.metadata!(df, "crs", crs_wkt; style=:note)
    DataFrames.metadata!(df, "GEOINTERFACE:crs", crs_wkt; style=:note)
    return df
end

In [ ]:
#| export
# TPI-based morphological high/low (GA-SaMMT "TPI Tool", Eq. 1-3): threshold
# TPI itself, above its mean+C*std for highs, below mean-C*std for lows.
function morphological_high(::TPIMethod, bathy::Bathymetry, radius::Real;
                             C::Real=1.0, area_threshold::Real=0.0, unit::AbstractString="meter")
    tpi_b = tpi(bathy, radius; unit=unit)
    clean_raster = replace_missing(tpi_b.raster, NaN)
    raw_matrix = Matrix{Float64}(Array(clean_raster))
    mask = threshold_mask(raw_matrix, C; sign=:above)
    return polygonize_filtered(mask, clean_raster, area_threshold)
end

function morphological_low(::TPIMethod, bathy::Bathymetry, radius::Real;
                            C::Real=1.0, area_threshold::Real=0.0, unit::AbstractString="meter")
    tpi_b = tpi(bathy, radius; unit=unit)
    clean_raster = replace_missing(tpi_b.raster, NaN)
    raw_matrix = Matrix{Float64}(Array(clean_raster))
    mask = threshold_mask(raw_matrix, C; sign=:below)
    return polygonize_filtered(mask, clean_raster, area_threshold)
end

In [ ]:
#| export
# Openness-based morphological high/low (GA-SaMMT "Openness Tool", Eq. 4-5):
# highs come from Negative Openness (smaller-than-mean NO = convex/high),
# lows come from Positive Openness (smaller-than-mean PO = concave/low) --
# both thresholds are "below mean - C*std", same as the paper.
function morphological_high(::OpennessMethod, bathy::Bathymetry, radius::Real;
                             C::Real=1.0, area_threshold::Real=0.0, unit::AbstractString="meter")
    no_b = openness(bathy, radius; unit=unit).negative
    clean_raster = replace_missing(no_b.raster, NaN)
    raw_matrix = Matrix{Float64}(Array(clean_raster))
    mask = threshold_mask(raw_matrix, C; sign=:below)
    return polygonize_filtered(mask, clean_raster, area_threshold)
end

function morphological_low(::OpennessMethod, bathy::Bathymetry, radius::Real;
                            C::Real=1.0, area_threshold::Real=0.0, unit::AbstractString="meter")
    po_b = openness(bathy, radius; unit=unit).positive
    clean_raster = replace_missing(po_b.raster, NaN)
    raw_matrix = Matrix{Float64}(Array(clean_raster))
    mask = threshold_mask(raw_matrix, C; sign=:below)
    return polygonize_filtered(mask, clean_raster, area_threshold)
end

In [ ]:
#| export
# BPI-based morphological high/low: BPI is center-minus-annulus-mean, the
# same "relative elevation" shape as TPI (Lundblad et al. 2006), just
# with an annulus kernel instead of a circle -- so it gets the identical
# threshold convention as TPIMethod (high = above mean+C*std, low = below
# mean-C*std). Not one of GA-SaMMT's four named Map-step tools (paper
# Figure 1 lists TPI/Openness/LMI/CI only, not BPI), but MIM_GA already
# has bpi() built and it fits this same mean+/-C*std rule cleanly, so it
# gets the same treatment as the others here.
function morphological_high(::BPIMethod, bathy::Bathymetry, inner_radius::Real, outer_radius::Real;
                             C::Real=1.0, area_threshold::Real=0.0, unit::AbstractString="meter")
    bpi_b = bpi(bathy, inner_radius, outer_radius; unit=unit)
    clean_raster = replace_missing(bpi_b.raster, NaN)
    raw_matrix = Matrix{Float64}(Array(clean_raster))
    mask = threshold_mask(raw_matrix, C; sign=:above)
    return polygonize_filtered(mask, clean_raster, area_threshold)
end

function morphological_low(::BPIMethod, bathy::Bathymetry, inner_radius::Real, outer_radius::Real;
                            C::Real=1.0, area_threshold::Real=0.0, unit::AbstractString="meter")
    bpi_b = bpi(bathy, inner_radius, outer_radius; unit=unit)
    clean_raster = replace_missing(bpi_b.raster, NaN)
    raw_matrix = Matrix{Float64}(Array(clean_raster))
    mask = threshold_mask(raw_matrix, C; sign=:below)
    return polygonize_filtered(mask, clean_raster, area_threshold)
end